# Post-Training — Foundations
> Section 01 — Topic 06 — foundations

**Prerequisites:** 05-pretraining

**What you'll learn:**
- Why post-training transforms base models into useful assistants
- How supervised fine-tuning (SFT) works end-to-end
- Chat templates and instruction dataset design

**What you'll build:**
- An SFT pipeline using TRL

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. Why Post-Training? — Base Models vs Instruction-Tuned vs Chat

A **base model** (also called a pretrained model) is trained on a massive text corpus with one objective: predict the next token. It learns grammar, facts, reasoning patterns, and even code — but it has no concept of following instructions. If you give it a question, it's equally likely to continue with another question, write an essay tangentially related to the topic, or produce gibberish. Base models are powerful pattern completors but terrible assistants.

**Post-training** is the process of taking a base model and transforming it into something that follows instructions, engages in helpful dialogue, and avoids harmful outputs. It typically involves two or three stages:

1. **Supervised Fine-Tuning (SFT):** Train on curated (instruction, response) pairs so the model learns to follow instructions and produce well-formatted responses.
2. **Preference Optimization (RLHF/DPO):** Train the model to prefer high-quality responses over mediocre ones using human preference data.
3. **Optional safety training:** Additional alignment to refuse harmful requests while remaining helpful.

The difference is dramatic. A base Llama model given "What is the capital of France?" might continue with "What is the capital of Germany? What is the capital of Spain?" (pattern completion). The instruction-tuned version responds: "The capital of France is Paris."

In [ ]:
# Compare base vs instruct model behavior
# Using GPT-2 as a base model example (no instruct version, but demonstrates the point)

base_model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
base_model.eval()

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

prompts = [
    "What is the capital of France?",
    "Write a Python function that adds two numbers.",
    "Explain quantum computing in simple terms.",
]

print("=== Base Model (GPT-2) — Pattern Completion ===")
print("Notice: it continues the text pattern, doesn't 'answer' the question\n")

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = base_model.generate(
            **inputs, max_new_tokens=50, do_sample=True, 
            temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Prompt: {prompt}")
    print(f"Output: {response[:200]}...")
    print()

## 2. Supervised Fine-Tuning (SFT) — From Scratch

SFT is conceptually simple: take your base model and train it on high-quality (instruction, response) pairs using the standard language modeling loss. The key subtlety is **loss masking** — you only compute the loss on the assistant's response tokens, not on the instruction/prompt tokens. This teaches the model to generate good responses given instructions, without wasting gradient updates on learning to reproduce the instructions themselves.

The training data format typically looks like:
```
<|system|>You are a helpful assistant.<|end|>
<|user|>What is photosynthesis?<|end|>
<|assistant|>Photosynthesis is the process by which plants convert sunlight into energy...<|end|>
```

The model sees the full sequence during training, but the loss is only computed on the tokens after `<|assistant|>`. This is crucial — without loss masking, the model wastes capacity memorizing prompts instead of learning to generate good completions.

In [ ]:
# Implement SFT training loop from scratch with loss masking

def create_sft_training_example(instruction, response, tokenizer, max_length=256):
    """
    Create a training example with proper loss masking.
    Loss is only computed on response tokens.
    """
    # Format the conversation
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    full_text = prompt + response + tokenizer.eos_token
    
    # Tokenize
    full_tokens = tokenizer(full_text, truncation=True, max_length=max_length, return_tensors="pt")
    prompt_tokens = tokenizer(prompt, return_tensors="pt")
    
    prompt_len = prompt_tokens.input_ids.shape[1]
    
    # Create labels: -100 for prompt tokens (ignored by CrossEntropyLoss)
    labels = full_tokens.input_ids.clone()
    labels[0, :prompt_len] = -100  # Mask prompt tokens
    
    return {
        'input_ids': full_tokens.input_ids,
        'attention_mask': full_tokens.attention_mask,
        'labels': labels,
    }

# Demo: show the masking
example = create_sft_training_example(
    "What is 2 + 2?",
    "2 + 2 equals 4.",
    tokenizer
)

tokens = tokenizer.convert_ids_to_tokens(example['input_ids'][0])
labels = example['labels'][0]

print("Token-by-token loss masking:")
print(f"{'Token':<15} {'Label':<10} {'Loss?'}")
print("-" * 35)
for tok, lab in zip(tokens, labels):
    loss_status = "MASKED" if lab == -100 else "COMPUTED"
    print(f"{tok:<15} {lab.item():<10} {loss_status}")

In [ ]:
# Full SFT training loop from scratch

def sft_training_step(model, batch, optimizer):
    """Single SFT training step with loss masking."""
    model.train()
    
    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    labels = batch['labels']
    
    # Forward pass
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss  # CrossEntropyLoss automatically ignores -100 labels
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    return loss.item()

# Create a small training dataset
training_data = [
    ("What is the capital of France?", "The capital of France is Paris. It is the largest city in France and serves as the country's political, economic, and cultural center."),
    ("Write hello world in Python.", "Here is a hello world program in Python:\n\nprint('Hello, World!')\n\nThis uses the built-in print function to output text to the console."),
    ("Explain what a neural network is.", "A neural network is a computational model inspired by biological neurons. It consists of layers of interconnected nodes that process information by passing signals through weighted connections. Through training, these weights are adjusted to minimize prediction errors."),
    ("What is 15 * 23?", "15 * 23 = 345. You can compute this by breaking it down: 15 * 20 = 300, and 15 * 3 = 45, so 300 + 45 = 345."),
    ("Summarize photosynthesis in one sentence.", "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen, converting light energy into chemical energy stored in sugar molecules."),
]

# Prepare batches
train_examples = [create_sft_training_example(inst, resp, tokenizer) for inst, resp in training_data]

# Train for a few steps (demo — real SFT uses thousands of examples)
model = AutoModelForCausalLM.from_pretrained("gpt2")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

losses = []
n_epochs = 5
for epoch in range(n_epochs):
    epoch_losses = []
    for ex in train_examples:
        loss = sft_training_step(model, ex, optimizer)
        epoch_losses.append(loss)
    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{n_epochs}: loss = {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, n_epochs+1), losses, 'o-')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('SFT Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Chat Templates — How Models Learn Turn Structure

Modern chat models use **chat templates** to structure multi-turn conversations. These templates define how system messages, user messages, and assistant responses are formatted before being fed to the model. Getting the template wrong can dramatically degrade model quality — the model was trained with a specific format and expects it at inference time.

Hugging Face tokenizers store chat templates as Jinja2 strings. The `apply_chat_template()` method renders a list of message dictionaries into the exact token sequence the model expects. Different model families use different templates — Llama 3 uses special `<|start_header_id|>` tokens, Mistral uses `[INST]` markers, and ChatML uses `<|im_start|>` / `<|im_end|>` tags.

Understanding these templates matters for production: if you're building a multi-turn chatbot, constructing few-shot prompts, or fine-tuning on conversation data, the template must match exactly.

In [ ]:
# Examine chat template structure

# ChatML format (used by many open models)
chatml_template = """{% for message in messages %}<|im_start|>{{ message['role'] }}
{{ message['content'] }}<|im_end|>
{% endfor %}<|im_start|>assistant
"""

# Example conversation
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "What is Python?"},
    {"role": "assistant", "content": "Python is a high-level programming language known for its readability and versatility."},
    {"role": "user", "content": "What are its main uses?"},
]

# Show how different templates format the same conversation
print("=== ChatML Format ===")
for msg in messages:
    print(f"<|im_start|>{msg['role']}")
    print(f"{msg['content']}<|im_end|>")
print("<|im_start|>assistant")

print("\n=== Llama 3 Format ===")
for msg in messages:
    print(f"<|start_header_id|>{msg['role']}<|end_header_id|>")
    print(f"\n{msg['content']}<|eot_id|>")
print("<|start_header_id|>assistant<|end_header_id|>")

print("\n=== Mistral/Instruct Format ===")
for i, msg in enumerate(messages):
    if msg['role'] == 'system':
        continue
    if msg['role'] == 'user':
        print(f"[INST] {msg['content']} [/INST]")
    else:
        print(f"{msg['content']}</s>")

In [ ]:
# Using HuggingFace's apply_chat_template
try:
    # Try loading a model with a chat template
    chat_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
    
    formatted = chat_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("Formatted conversation (Zephyr template):")
    print(formatted)
    
    # Show token IDs for special tokens
    token_ids = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    print(f"\nTotal tokens: {len(token_ids)}")
    
except Exception as e:
    print(f"Could not load Zephyr tokenizer: {e}")
    print("Using GPT-2 tokenizer without chat template as fallback.")
    print("\nIn practice, you'd use a model that has a chat_template defined.")
    print("Key point: always use apply_chat_template() rather than formatting manually.")

## 4. Instruction Dataset Design

The quality of your SFT dataset is the single biggest factor in the resulting model's quality. A common misconception is that more data is always better — in practice, **10,000 high-quality, diverse examples often outperform 1,000,000 noisy examples**. The LIMA paper (Zhou et al., 2023) demonstrated that just 1,000 carefully curated examples could produce a surprisingly strong instruction-following model.

Key principles for instruction dataset design:

- **Diversity:** Cover many task types (QA, summarization, coding, math, creative writing, analysis). A model trained only on QA pairs will struggle with open-ended generation.
- **Quality:** Each response should be the kind of response you'd want the model to produce. Errors in training data become errors in the model.
- **Difficulty gradient:** Include easy, medium, and hard examples. Only training on hard examples can make the model overthink simple questions.
- **Format variety:** Mix single-turn and multi-turn, short and long responses, structured and unstructured outputs.

In [ ]:
# Analyze popular instruction datasets

datasets_info = {
    "Stanford Alpaca": {
        "size": "52K",
        "source": "GPT-3.5 generated from 175 seed tasks",
        "quality": "Medium — generated, some errors",
        "diversity": "Good — wide task range",
        "turns": "Single-turn only",
    },
    "OpenHermes 2.5": {
        "size": "1M+",
        "source": "Multiple sources, GPT-4 generated",
        "quality": "High — GPT-4 responses",
        "diversity": "Excellent — code, math, creative, factual",
        "turns": "Mostly single-turn",
    },
    "UltraChat": {
        "size": "1.5M conversations",
        "source": "GPT-3.5/4 multi-turn simulated conversations",
        "quality": "Good — natural conversations",
        "diversity": "Excellent — 30+ topic categories",
        "turns": "Multi-turn (3-10 turns)",
    },
    "LIMA": {
        "size": "1K",
        "source": "Manually curated from StackExchange, wikiHow, Reddit",
        "quality": "Very high — human-written, carefully selected",
        "diversity": "Good for size",
        "turns": "Single-turn",
    },
}

print(f"{'Dataset':<20} {'Size':<12} {'Quality':<35} {'Turns':<20}")
print("=" * 90)
for name, info in datasets_info.items():
    print(f"{name:<20} {info['size']:<12} {info['quality']:<35} {info['turns']:<20}")

print("\nKey takeaway: LIMA (1K examples) showed that quality >> quantity.")
print("Start with small, high-quality data. Scale up only if needed.")

In [ ]:
# Build quality indicators for an instruction dataset

def analyze_instruction_dataset(examples):
    """Compute quality metrics for an instruction dataset."""
    metrics = {
        'n_examples': len(examples),
        'avg_instruction_len': np.mean([len(inst.split()) for inst, _ in examples]),
        'avg_response_len': np.mean([len(resp.split()) for _, resp in examples]),
        'instruction_len_std': np.std([len(inst.split()) for inst, _ in examples]),
        'response_len_std': np.std([len(resp.split()) for _, resp in examples]),
        'min_response_len': min(len(resp.split()) for _, resp in examples),
        'max_response_len': max(len(resp.split()) for _, resp in examples),
    }
    
    # Detect potential issues
    issues = []
    short_responses = sum(1 for _, r in examples if len(r.split()) < 5)
    if short_responses > 0:
        issues.append(f"{short_responses} responses < 5 words (too short?)")
    
    duplicates = len(examples) - len(set(inst for inst, _ in examples))
    if duplicates > 0:
        issues.append(f"{duplicates} duplicate instructions")
    
    metrics['issues'] = issues
    return metrics

# Analyze our training data
stats = analyze_instruction_dataset(training_data)
print("Dataset Statistics:")
for key, val in stats.items():
    if key != 'issues':
        print(f"  {key}: {val:.1f}" if isinstance(val, float) else f"  {key}: {val}")

if stats['issues']:
    print("\nPotential Issues:")
    for issue in stats['issues']:
        print(f"  ⚠️ {issue}")
else:
    print("\n  ✅ No obvious issues detected")

# Visualize length distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
inst_lens = [len(inst.split()) for inst, _ in training_data]
resp_lens = [len(resp.split()) for _, resp in training_data]

ax1.bar(range(len(inst_lens)), inst_lens, color='steelblue')
ax1.set_title('Instruction Lengths (words)')
ax1.set_xlabel('Example')

ax2.bar(range(len(resp_lens)), resp_lens, color='coral')
ax2.set_title('Response Lengths (words)')
ax2.set_xlabel('Example')

plt.tight_layout()
plt.show()

## 5. SFT in Practice — Using TRL

While it's valuable to understand SFT from scratch (as we did above), in practice you'll use the **TRL (Transformer Reinforcement Learning)** library from Hugging Face. TRL's `SFTTrainer` handles all the complexity: dataset formatting, loss masking, chat template application, gradient accumulation, mixed precision, logging, and checkpointing.

The typical workflow is:
1. Load a base model and tokenizer
2. Prepare your dataset in the expected format (list of messages)
3. Configure `SFTConfig` with training hyperparameters
4. Create an `SFTTrainer` and call `.train()`

TRL also integrates with PEFT (Parameter-Efficient Fine-Tuning) for LoRA/QLoRA, which we'll cover in Section 12.

In [ ]:
# SFT with TRL — production-ready approach
try:
    from trl import SFTConfig, SFTTrainer
    from datasets import Dataset
    
    # Prepare dataset in conversation format
    conversations = []
    for inst, resp in training_data:
        conversations.append({
            "messages": [
                {"role": "user", "content": inst},
                {"role": "assistant", "content": resp},
            ]
        })
    
    dataset = Dataset.from_list(conversations)
    print(f"Dataset: {len(dataset)} examples")
    print(f"Example: {dataset[0]}")
    
    # Configure SFT training
    sft_config = SFTConfig(
        output_dir="./sft_output",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=10,
        logging_steps=1,
        max_seq_length=256,
        save_strategy="no",  # Don't save checkpoints for demo
    )
    
    # Note: In practice you'd use a proper base model like TinyLlama
    # Here we use GPT-2 for demonstration
    print("\nSFTConfig created. In production:")
    print("  trainer = SFTTrainer(model=model, args=sft_config, train_dataset=dataset)")
    print("  trainer.train()")
    
except ImportError:
    print("Install TRL: pip install trl")
    print("\nTRL provides SFTTrainer which handles:")
    print("  - Automatic chat template application")
    print("  - Loss masking on prompt tokens")
    print("  - Mixed precision training")
    print("  - Gradient accumulation")
    print("  - Integration with PEFT/LoRA")

In [ ]:
# Compare outputs: before vs after SFT (using our from-scratch trained model)
model.eval()

test_prompts = [
    "### Instruction:\nWhat is machine learning?\n\n### Response:\n",
    "### Instruction:\nWrite a greeting in Python.\n\n### Response:\n",
]

print("=== After SFT (few examples, demonstrative) ===")
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=60, do_sample=True,
            temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Prompt: {prompt.strip()[:50]}...")
    print(f"Response: {response[:150]}")
    print()

print("Note: With only 5 training examples, quality is limited.")
print("Real SFT uses 10K-100K high-quality examples for strong results.")

## Summary

**Key takeaways:**
- Base models are pattern completors; post-training transforms them into instruction followers
- SFT trains on (instruction, response) pairs with **loss masking** on prompt tokens
- Chat templates (ChatML, Llama, Mistral) define how conversations are formatted — must match training format
- Dataset quality matters more than quantity — 1K curated examples (LIMA) can outperform 1M noisy ones
- TRL's SFTTrainer is the production tool — handles formatting, masking, PEFT integration

**Next:** [06-post-training/advanced.ipynb](advanced.ipynb) — Reward modeling, RLHF with PPO, DPO